# C/C++ full retrain â€” CodeBERT + GraphCodeBERT + Hybrid (corrected labels)

Runs the three real, controlled, full-scale retrains that were never
actually finished (the previous background CPU job died after ~1 hour on
only a 4,000-row debug subset). Trains, in order:

1. `src/models/train_codebert_full.py` -> `results/codebert_evaluation.AFTER_label_fix.json`
2. `src/models/train_graphcodebert_renku.py` -> `results/graphcodebert_evaluation.AFTER_label_fix.json`
3. `src/models/train_hybrid_full.py` -> `results/hybrid_evaluation.AFTER_label_fix.json`

All three train on the full, corrected `data/splits/{train,val,test}_dedup.parquet`
splits (73,295 / ~9,162 train/test rows) â€” not a subsample.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine).

**Before running:** make sure `cpp_full_retrain_bundle.zip` exists at the
repo root (bundles the 3 scripts + 3 parquet splits, ~9 MB). Recreate it
from the repo root if needed:

```python
import zipfile, os
files_to_zip = [
    "src/models/train_codebert_full.py",
    "src/models/train_graphcodebert_renku.py",
    "src/models/train_hybrid_full.py",
    "data/splits/train_dedup.parquet",
    "data/splits/val_dedup.parquet",
    "data/splits/test_dedup.parquet",
]
with zipfile.ZipFile("cpp_full_retrain_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files_to_zip:
        z.write(f, arcname=f)
```

## Step 1 â€” upload cpp_full_retrain_bundle.zip

Run this cell, then click **Choose Files** and select `cpp_full_retrain_bundle.zip`.

In [ ]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one file: cpp_full_retrain_bundle.zip"
zip_name = next(iter(uploaded))
print("Uploaded:", zip_name, f"({len(uploaded[zip_name]) / 1e6:.1f} MB)")

## Step 2 â€” unpack into the session's local disk

In [ ]:
import zipfile, os

WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)

with zipfile.ZipFile(zip_name) as z:
    z.extractall(WORK_DIR)

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

## Step 3 â€” preflight: verify files unpacked correctly

In [ ]:
required = [
    "src/models/train_codebert_full.py",
    "src/models/train_graphcodebert_renku.py",
    "src/models/train_hybrid_full.py",
    "data/splits/train_dedup.parquet",
    "data/splits/val_dedup.parquet",
    "data/splits/test_dedup.parquet",
]

missing = [p for p in required if not os.path.isfile(p)]

for p in required:
    ok = os.path.isfile(p)
    size = f"{os.path.getsize(p) / 1e6:.1f} MB" if ok else "MISSING"
    print(f"{'OK ' if ok else 'XX '} {p}  {size}")

assert not missing, (
    f"{len(missing)} required file(s) missing after unzip -- rebuild "
    "cpp_full_retrain_bundle.zip and re-upload."
)

## Step 4 â€” install dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn pandas pyarrow tqdm

Colab's preinstalled `torchvision` is incompatible with the `datasets` library's tensor formatter (unconditional `from torchvision.io import VideoReader` that crashes even for pure-text data â€” hit this exact bug during the Java/C# runs). Not needed for text classification, so remove it.

In [ ]:
!pip uninstall -y -q torchvision torchaudio

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (training will be slow -- enable a GPU runtime)")

## Step 5 â€” train CodeBERT (full)

Uses `subprocess.run(..., check=True)` so a failing script raises a real exception with the full traceback instead of silently reporting success.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "src/models/train_codebert_full.py"],
    check=True,
)
print("CodeBERT training finished with exit code", result.returncode)

### Safety net — download CodeBERT (C/C++) now

Run this immediately after the step above finishes, before starting the next model. If the session dies or runs out of compute later, you keep this result instead of losing the whole notebook's progress.

In [ ]:
import os, zipfile

_ckpt = "experiments/codebert-full"
_res = "results/codebert_evaluation.AFTER_label_fix.json"
_name = "safety_codebert_cpp"
if os.path.isdir(_ckpt):
    exclude = {"optimizer.pt", "scheduler.pt", "rng_state.pth"}
    with zipfile.ZipFile(f"{_name}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _dirs, fnames in os.walk(_ckpt):
            for fn in fnames:
                if fn in exclude:
                    continue
                fp = os.path.join(root, fn)
                zf.write(fp, os.path.relpath(fp, os.path.dirname(_ckpt)))
    print(f"{_name}.zip -> {os.path.getsize(_name + '.zip') / 1e6:.1f} MB")
    files.download(f"{_name}.zip")
else:
    print("No checkpoint dir at", _ckpt, "-- check the training step above succeeded.")
if os.path.isfile(_res):
    files.download(_res)
else:
    print("No results file at", _res, "-- check the training step above succeeded.")


## Step 6 â€” train GraphCodeBERT (full)

In [ ]:
result = subprocess.run(
    [sys.executable, "src/models/train_graphcodebert_renku.py"],
    check=True,
)
print("GraphCodeBERT training finished with exit code", result.returncode)

### Safety net — download GraphCodeBERT (C/C++) now

Run this immediately after the step above finishes, before starting the next model. If the session dies or runs out of compute later, you keep this result instead of losing the whole notebook's progress.

In [ ]:
import os, zipfile

_ckpt = "experiments/graphcodebert-full"
_res = "results/graphcodebert_evaluation.AFTER_label_fix.json"
_name = "safety_graphcodebert_cpp"
if os.path.isdir(_ckpt):
    exclude = {"optimizer.pt", "scheduler.pt", "rng_state.pth"}
    with zipfile.ZipFile(f"{_name}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _dirs, fnames in os.walk(_ckpt):
            for fn in fnames:
                if fn in exclude:
                    continue
                fp = os.path.join(root, fn)
                zf.write(fp, os.path.relpath(fp, os.path.dirname(_ckpt)))
    print(f"{_name}.zip -> {os.path.getsize(_name + '.zip') / 1e6:.1f} MB")
    files.download(f"{_name}.zip")
else:
    print("No checkpoint dir at", _ckpt, "-- check the training step above succeeded.")
if os.path.isfile(_res):
    files.download(_res)
else:
    print("No results file at", _res, "-- check the training step above succeeded.")


## Step 7 â€” train Hybrid (full)

This one trains a classifier head on top of two frozen backbones (CodeBERT + GraphCodeBERT) â€” heavier than the other two since it runs both models' forward pass per batch.

In [ ]:
result = subprocess.run(
    [sys.executable, "src/models/train_hybrid_full.py"],
    check=True,
)
print("Hybrid training finished with exit code", result.returncode)

### Safety net — download Hybrid (C/C++) now

Run this immediately after the step above finishes, before starting the next model. If the session dies or runs out of compute later, you keep this result instead of losing the whole notebook's progress.

In [ ]:
import os, zipfile

_ckpt = "experiments/hybrid-full"
_res = "results/hybrid_evaluation.AFTER_label_fix.json"
_name = "safety_hybrid_cpp"
if os.path.isdir(_ckpt):
    exclude = {"optimizer.pt", "scheduler.pt", "rng_state.pth"}
    with zipfile.ZipFile(f"{_name}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _dirs, fnames in os.walk(_ckpt):
            for fn in fnames:
                if fn in exclude:
                    continue
                fp = os.path.join(root, fn)
                zf.write(fp, os.path.relpath(fp, os.path.dirname(_ckpt)))
    print(f"{_name}.zip -> {os.path.getsize(_name + '.zip') / 1e6:.1f} MB")
    files.download(f"{_name}.zip")
else:
    print("No checkpoint dir at", _ckpt, "-- check the training step above succeeded.")
if os.path.isfile(_res):
    files.download(_res)
else:
    print("No results file at", _res, "-- check the training step above succeeded.")


## Step 8 â€” view results

In [ ]:
import json

for path in [
    "results/codebert_evaluation.AFTER_label_fix.json",
    "results/graphcodebert_evaluation.AFTER_label_fix.json",
    "results/hybrid_evaluation.AFTER_label_fix.json",
]:
    with open(path) as f:
        print(path)
        print(json.dumps(json.load(f), indent=2))
        print()

## Step 9 â€” download results + checkpoints back to your machine

Colab's local disk is wiped when the runtime ends, so zip up `results/` and `experiments/` and download them now.

In [ ]:
import shutil

shutil.make_archive("cpp_full_retrain_output", "zip", WORK_DIR, "results")
shutil.make_archive("cpp_full_retrain_checkpoints", "zip", WORK_DIR, "experiments")

files.download("cpp_full_retrain_output.zip")
files.download("cpp_full_retrain_checkpoints.zip")